In [1]:
import pandas as pd
import ast

original_data = pd.read_csv('/home/s6moakba/InstructABSA/Dataset/SemEval14/Train/Restaurants_Train.csv')
original_data.head()

,sentenceId,raw_text,aspectTerms,aspectCategories
0,3121,But the staff was so horrible to us.,"[{'term': 'staff', 'polarity': 'negative'}]","[{'category': 'service', 'polarity': 'negative'}]"
1,2777,"To be completely fair, the only redeeming fact...","[{'term': 'food', 'polarity': 'positive'}]","[{'category': 'food', 'polarity': 'positive'},..."
2,1634,"The food is uniformly exceptional, with a very...","[{'term': 'food', 'polarity': 'positive'}, {'t...","[{'category': 'food', 'polarity': 'positive'}]"
3,2534,Where Gabriela personaly greets you and recomm...,"[{'term': 'noaspectterm', 'polarity': 'none'}]","[{'category': 'service', 'polarity': 'positive'}]"
4,583,"For those that go once and don't enjoy it, all...","[{'term': 'noaspectterm', 'polarity': 'none'}]","[{'category': 'anecdotes/miscellaneous', 'pola..."


In [2]:
gen_data = pd.read_csv('/home/s6moakba/Thesis/agent_practice/qwen_generate_14_final_laptop.csv')

In [3]:
gen_data

,Terms,Polarity,prompt,text,sentence
0,"charged,cad programs","negative,negative",\n You are a critic who can gen...,"content=""The battery doesn't charge properly, ...","The battery doesn't charge properly, and the C..."
1,drag and drop feature,negative,\n You are a critic who can gen...,"content=""The restaurant's drag and drop featur...",The restaurant's drag and drop feature for ord...
2,"warranty service to Toshiba,Powerpoint program","negative,positive",\n You are a critic who can gen...,"content=""The warranty service from Toshiba was...",The warranty service from Toshiba was disappoi...
3,Battery Life,positive,\n You are a critic who can gen...,"content=""The restaurant offers excellent batte...",The restaurant offers excellent battery life f...
4,"service tech,VHS","positive,negative",\n You are a critic who can gen...,"content=""The restaurant offers excellent servi...",The restaurant offers excellent service but st...
...,...,...,...,...,...
5395,"Drivers/Applications DVD,start up","positive,neutral",\n You are a critic who can gen...,"content=""The provided Drivers/Applications DVD...",The provided Drivers/Applications DVD ensures ...
5396,costed,negative,\n You are a critic who can gen...,"content=""The meal costed more than expected, l...","The meal costed more than expected, leaving a ..."
5397,sales tax,positive,\n You are a critic who can gen...,"content=""The restaurant offers a competitive s...",The restaurant offers a competitive sales tax ...
5398,"looking,user interface","negative,positive",\n You are a critic who can gen...,"content=""The restaurant has a confusing layout...",The restaurant has a confusing layout when you...


In [4]:
gen_data.columns.tolist()

['Terms', 'Polarity', 'prompt', 'text', 'sentence']

In [5]:
gen_data['Terms'] = gen_data['Terms'].apply(lambda x: x.split(','))
gen_data['Polarity'] = gen_data['Polarity'].apply(lambda x: x.split(','))

In [ ]:
# def process_terms(terms):
#     terms_list = terms[1:-1].split(', ')
#     terms_list = [item[1:-1].replace("\\", "") for item in terms_list]
#     return terms_list

In [ ]:
# gen_data['Terms'] = gen_data['Terms'].apply(process_terms)
# gen_data['Polarity'] = gen_data['Polarity'].apply(process_terms)

In [ ]:
# mask_no_backslash = ~gen_data.apply(
#     lambda row: row.astype(str).str.contains(r'\\', regex=True).any(),
#     axis=1
# )
# gen_data = gen_data[mask_no_backslash].reset_index(drop=True)

In [7]:
gen_data['sentence'] = gen_data['sentence'].apply(lambda x: x if isinstance(x, str) else '.')

In [8]:
gen_data

,Terms,Polarity,prompt,text,sentence
0,"[charged, cad programs]","[negative, negative]",\n You are a critic who can gen...,"content=""The battery doesn't charge properly, ...","The battery doesn't charge properly, and the C..."
1,[drag and drop feature],[negative],\n You are a critic who can gen...,"content=""The restaurant's drag and drop featur...",The restaurant's drag and drop feature for ord...
2,"[warranty service to Toshiba, Powerpoint program]","[negative, positive]",\n You are a critic who can gen...,"content=""The warranty service from Toshiba was...",The warranty service from Toshiba was disappoi...
3,[Battery Life],[positive],\n You are a critic who can gen...,"content=""The restaurant offers excellent batte...",The restaurant offers excellent battery life f...
4,"[service tech, VHS]","[positive, negative]",\n You are a critic who can gen...,"content=""The restaurant offers excellent servi...",The restaurant offers excellent service but st...
...,...,...,...,...,...
5395,"[Drivers/Applications DVD, start up]","[positive, neutral]",\n You are a critic who can gen...,"content=""The provided Drivers/Applications DVD...",The provided Drivers/Applications DVD ensures ...
5396,[costed],[negative],\n You are a critic who can gen...,"content=""The meal costed more than expected, l...","The meal costed more than expected, leaving a ..."
5397,[sales tax],[positive],\n You are a critic who can gen...,"content=""The restaurant offers a competitive s...",The restaurant offers a competitive sales tax ...
5398,"[looking, user interface]","[negative, positive]",\n You are a critic who can gen...,"content=""The restaurant has a confusing layout...",The restaurant has a confusing layout when you...


# filter rows that do not contain Terms

In [48]:
def filter_rows(row):
    aspects = row['Terms']
    if all(aspect in row['sentence'] for aspect in aspects):
        row['correct'] = True    
    else:
        row['correct'] = False
    return row

gen_data = gen_data.apply(filter_rows, axis=1)  

In [49]:
gen_data_filtered = gen_data[gen_data['correct'] == True]
gen_data_filtered.shape

(5293, 4)

In [50]:
gen_data_filtered = gen_data_filtered.drop(columns=['correct'])

In [9]:
def create_aspect_terms(aspect,polarity):
    aspect_terms = []
    for aspect, polarity in zip(aspect, polarity):
        aspect_terms.append({'term': aspect, 'polarity': polarity})
    return aspect_terms 

In [52]:
gen_data_filtered['aspectTerms'] = gen_data_filtered.apply(lambda x: create_aspect_terms(x['Terms'],x['Polarity']),axis=1)

In [10]:
gen_data['aspectTerms'] = gen_data.apply(lambda x: create_aspect_terms(x['Terms'],x['Polarity']),axis=1)

In [11]:
print(gen_data['aspectTerms'].iloc[20])
print(gen_data['sentence'].iloc[20])

[{'term': 'CD drive', 'polarity': 'negative'}, {'term': 'responds', 'polarity': 'positive'}, {'term': 'product and help aftermarket', 'polarity': 'positive'}]
The CD drive is frustratingly slow to respond, but the product quality and after-sales support are excellent


In [55]:
gen_data_filtered.rename(columns={'sentence':'raw_text'},inplace=True)

In [56]:
gen_data_filtered['aspectCategories'] = gen_data_filtered['aspectTerms'].apply(lambda x: [{'category': 'general', 'polarity': 'neutral'}])

In [57]:
gen_data_filtered['sentenceId'] = gen_data_filtered.index

In [12]:
gen_data.rename(columns={'sentence':'raw_text'},inplace=True)
gen_data['aspectCategories'] = gen_data['aspectTerms'].apply(lambda x: [{'category': 'general', 'polarity': 'neutral'}])
gen_data.drop(columns=['text','prompt'], inplace=True)
gen_data['sentenceId'] = gen_data.index

In [13]:
gen_data

,Terms,Polarity,raw_text,aspectTerms,aspectCategories,sentenceId
0,"[charged, cad programs]","[negative, negative]","The battery doesn't charge properly, and the C...","[{'term': 'charged', 'polarity': 'negative'}, ...","[{'category': 'general', 'polarity': 'neutral'}]",0
1,[drag and drop feature],[negative],The restaurant's drag and drop feature for ord...,"[{'term': 'drag and drop feature', 'polarity':...","[{'category': 'general', 'polarity': 'neutral'}]",1
2,"[warranty service to Toshiba, Powerpoint program]","[negative, positive]",The warranty service from Toshiba was disappoi...,"[{'term': 'warranty service to Toshiba', 'pola...","[{'category': 'general', 'polarity': 'neutral'}]",2
3,[Battery Life],[positive],The restaurant offers excellent battery life f...,"[{'term': 'Battery Life', 'polarity': 'positiv...","[{'category': 'general', 'polarity': 'neutral'}]",3
4,"[service tech, VHS]","[positive, negative]",The restaurant offers excellent service but st...,"[{'term': 'service tech', 'polarity': 'positiv...","[{'category': 'general', 'polarity': 'neutral'}]",4
...,...,...,...,...,...,...
5395,"[Drivers/Applications DVD, start up]","[positive, neutral]",The provided Drivers/Applications DVD ensures ...,"[{'term': 'Drivers/Applications DVD', 'polarit...","[{'category': 'general', 'polarity': 'neutral'}]",5395
5396,[costed],[negative],"The meal costed more than expected, leaving a ...","[{'term': 'costed', 'polarity': 'negative'}]","[{'category': 'general', 'polarity': 'neutral'}]",5396
5397,[sales tax],[positive],The restaurant offers a competitive sales tax ...,"[{'term': 'sales tax', 'polarity': 'positive'}]","[{'category': 'general', 'polarity': 'neutral'}]",5397
5398,"[looking, user interface]","[negative, positive]",The restaurant has a confusing layout when you...,"[{'term': 'looking', 'polarity': 'negative'}, ...","[{'category': 'general', 'polarity': 'neutral'}]",5398


In [15]:
path = '/home/s6moakba/InstructABSA/Dataset/AUG/14_l/prompting_14_l_5k.csv'

gen_data.to_csv(path, index=False)